In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import json
import matplotlib as mpl
mpl.rcParams["font.family"] = "monospace"

from plotting import plot_model_comparison

In [ ]:
dataset_name = "crc" # Options: merfish, crc

In [ ]:
corr_dir = f"/data2/a330d/datasets/{dataset_name}/correlations"
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    holdout_celltype = "_".join(parts[2:])
    try:
        with open(fp, "r") as f:
            data = json.load(f)
    except Exception:
        # skip unreadable/invalid json
        continue

    try:
        rows.append({
            "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
            "model_name": model_name,
            "holdout_celltype": holdout_celltype,
            "n_deg": data.get("n_deg"),
            "spearman": data.get("spearman"),
            "pearson": data.get("pearson"),
            "precision": data.get("precision"),
            "direction_match": data.get("direction_match"),
            "direction_match_k": data.get("direction_match_k"),
            "mixing_index": data.get("mixing_index"),
            "edistance_global": data.get("edistance_global"),
            "edistance_local": data.get("edistance_local"),            
            "rmse": data.get("rmse"),
        })
    except Exception as e:
        print(f"Error processing file {fp}: {e}")
        continue

data_df = pd.DataFrame(rows)
data_df.head()

In [ ]:
# Remove -cf from the end of each model_name
data_df["model_name"] = data_df["model_name"].str.replace("-cf", "", regex=False)
n_deg = data_df["n_deg"].iloc[0]
data_df.head()

In [ ]:
df = data_df.copy() # start with existing dataframe
df["sid"] = df["sid"].astype(str)

In [ ]:
def unify_formatting(df_parent, df_child):
    df_child["holdout_celltype"] = df_child["holdout_celltype"] + "_" + df_child["target_domain"]
    df_child["sid"] = df_child["sid"].astype(str)

    common_cols = df_parent.columns
    df_child_trimmed = df_child[common_cols.intersection(df_child.columns)]
    combined = pd.concat([df_parent, df_child_trimmed], axis=0, ignore_index=True)

    return combined

In [ ]:
# Load mintflow results
mintflow_base_path = f"/data/a330d/projects/cellina-reproducibility-worktrees/major-loo/results/mintflow_{dataset_name}"
mintflow_df = pd.DataFrame()

# Loop over mintflow_base_path and read each file, concatenate into mintflow_df
for file in glob.glob(f"{mintflow_base_path}/mintflow_*.csv"):
    temp_df = pd.read_csv(file)
    mintflow_df = pd.concat([mintflow_df, temp_df], ignore_index=True)
    mintflow_df = mintflow_df[mintflow_df["n_deg"] == n_deg]
mintflow_df.head()

In [ ]:
combined_df = unify_formatting(df, mintflow_df)

In [ ]:
spprop_csv_path = f'../results/loo_spatialprop_{dataset_name}_DEG_{int(n_deg)}.csv'
spprop_df = pd.read_csv(spprop_csv_path)
spprop_df.head()

In [ ]:
combined_df = unify_formatting(combined_df, spprop_df)

In [ ]:
# Add node-perturbation results
cellina_pert_df = pd.read_csv(f"../results/loo_cellina_{dataset_name}_DEG_{int(n_deg)}_pert.csv")
spprop_pert_df = pd.read_csv(f"../results/loo_spatialprop_{dataset_name}_DEG_{int(n_deg)}_pert.csv")
spprop_pert_df['model_name'] += "-pert"

In [ ]:
combined_df = unify_formatting(combined_df, cellina_pert_df)
combined_df = unify_formatting(combined_df, spprop_pert_df)

In [ ]:
grouping = "holdout_celltype"

# Define order
model_order = [
    "baseline",
    "mintflow",
    "spatialprop",
    "cpa",
    "scgen",
    "cellina-ablated",
    "cellina-graph",
    "cellina",
    "spatialprop-pert",
    "cellina-pert"
]

default_palette = 'colorblind'
palette = default_palette

In [ ]:
# Base palette (colorblind-friendly)
base_colors = sns.color_palette(default_palette, n_colors=len(model_order))

# Convert to dict mapped by model name
palette_dict = dict(zip(model_order, base_colors))

# Create a red gradient for the 3 cellina models
cellina_reds = sns.color_palette("Reds", 5)[1:] 

# skip the lightest so it's still visible
palette_dict["cellina-ablated"] = cellina_reds[1]
palette_dict["cellina-graph"]   = cellina_reds[2]
palette_dict["cellina"]         = cellina_reds[3]
palette_dict["cellina-pert"]         = cellina_reds[3]

# non-spatial green
palette_dict["mintflow"]   = sns.color_palette("Greens", 4)[1]
palette_dict["scgen"]   = sns.color_palette("Greens", 4)[2]
palette_dict["cpa"] = sns.color_palette("Greens", 4)[3]
palette_dict["spatialprop"]   = sns.color_palette("Greens", 4)[0]
palette_dict["spatialprop-pert"]   = sns.color_palette("Greens", 4)[0]

palette = palette_dict

In [ ]:
metrics = ["pearson", "spearman", "precision", "direction_match_k"]
plot_model_comparison(
    dataset_name,
    combined_df,
    metrics,
    grouping,
    model_order,
    palette,
    average_only=False,
    save=False,
    save_name=f"ood_summary_correlations_{dataset_name}_{n_deg}"
)

In [ ]:
metrics = ["edistance_global", "edistance_local"]
plot_model_comparison(
    dataset_name,
    combined_df,
    metrics,
    grouping,
    model_order,
    palette,
    average_only=False,
    save=False,
    save_name=f"ood_summary_correlations_{dataset_name}_{n_deg}"
)

In [ ]:
metrics = ["rmse"]
plot_model_comparison(
    dataset_name,
    combined_df,
    metrics,
    grouping,
    model_order,
    palette,
    average_only=False,
    save=False,
    save_name=f"ood_summary_correlations_{dataset_name}_{n_deg}"
)

In [ ]:
combined_df.to_csv(f"../results/loo_summary_{dataset_name}_DEG_{n_deg}.csv", index=False)

## Add average rank column

In [ ]:
metrics = [
    "pearson",
    #"spearman",
    "precision",
    #"direction_match",
    #"rmse",
    "edistance_local",
]

direction = {
    "pearson": "higher",
    #"spearman": "higher",
    "precision": "higher",
    #"direction_match": "higher",
    #"rmse": "lower",
    "edistance_local": "lower",
}

In [ ]:
df_ranked = combined_df.copy()

# compute ranks only temporarily
rank_cols = []
group_cols = ["sid", "holdout_celltype"]

for m in metrics:
    ascending = direction[m] == "lower"

    rank_col = f"{m}_rank"
    df_ranked[rank_col] = (
        df_ranked.groupby(group_cols)[m]
        .rank(ascending=ascending, method="average")
    )
    rank_cols.append(rank_col)

# final aggregate rank
df_ranked["avg_rank"] = df_ranked[rank_cols].mean(axis=1)

# drop intermediate rank columns
df_ranked = df_ranked.drop(columns=rank_cols)

In [ ]:
df_ranked

In [ ]:
df_ranked.to_csv(f"../results/loo_summary_{dataset_name}_DEG_{n_deg}_ranked.csv", index=False)